<a href="https://colab.research.google.com/github/mab0bakar/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mab0bakar/Run-the-Starter-Notebooks/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
from datasets import load_dataset

content = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_content",
    split="train[:100]"
)

print(content.column_names)

['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule:

Prioritize content for refresh when it has not been updated recently and appears stale.

The score should increase when:
- Content has not been updated for a long time.
- Content is older.
- Content is eligible for optimization.

Action Label:

REFRESH_CONTENT

Reason Codes:

- STALE_CONTENT
- OLD_CONTENT
- OPTIMIZATION_ELIGIBLE

Why this rule?

These signals are available before making a refresh decision and do not rely on future outcomes.

In [11]:
from datasets import load_dataset
import pandas as pd

content = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_content",
    split="train[:5000]"
)

content_df = pd.DataFrame(content)

print("Rows:", len(content_df))
print(content_df.columns.tolist())

Rows: 5000
['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']


In [9]:
content_df.columns.tolist()

['client_hash_id',
 'content_hash_id',
 'keyword_hash_id',
 'url_hash_id',
 'keyword_char_count',
 'keyword_token_count',
 'url_char_count',
 'content_created_date',
 'content_updated_date',
 'content_type',
 'search_volume',
 'competition',
 'competition_level',
 'cpc',
 'main_intent',
 'backlinks',
 'category_count',
 'keyword_created_date',
 'provider_used',
 'model_used',
 'char_count',
 'word_count',
 'last_optimized_date',
 'optimization_eligible_date',
 'is_published',
 'is_deleted']

#Signal Check Code

In [10]:
content_df["content_updated_date"] = pd.to_datetime(
    content_df["content_updated_date"]
)

latest_date = content_df["content_updated_date"].max()

content_df["days_since_update"] = (
    latest_date - content_df["content_updated_date"]
).dt.days

content_df["update_bucket"] = pd.cut(
    content_df["days_since_update"],
    bins=[0, 90, 180, 365, 9999],
    labels=["0-3m", "3-6m", "6-12m", "1y+"]
)

signal1 = (
    content_df.groupby("update_bucket")
    .agg(
        n=("content_hash_id", "count")
    )
)

signal1

/tmp/ipykernel_1001/1538047491.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  content_df.groupby("update_bucket")


,n
update_bucket,
0-3m,4932
3-6m,0
6-12m,0
1y+,0


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## Build the Ranked Queue

The baseline score combines:

1. Days since last update
2. Content age
3. Optimization eligibility

Higher scores indicate higher refresh priority.

The output is a ranked queue with:

- baseline_score
- reason_code
- action_label

In [12]:
import pandas as pd
import os

content_df["content_updated_date"] = pd.to_datetime(
    content_df["content_updated_date"]
)

content_df["content_created_date"] = pd.to_datetime(
    content_df["content_created_date"]
)

latest_date = content_df["content_updated_date"].max()

content_df["days_since_update"] = (
    latest_date - content_df["content_updated_date"]
).dt.days

content_df["content_age_days"] = (
    latest_date - content_df["content_created_date"]
).dt.days

content_df["optimization_flag"] = (
    content_df["optimization_eligible_date"].notna()
).astype(int)

content_df["baseline_score"] = (
    0.4 * (
        content_df["days_since_update"]
        / content_df["days_since_update"].max()
    )
    +
    0.4 * (
        content_df["content_age_days"]
        / content_df["content_age_days"].max()
    )
    +
    0.2 * content_df["optimization_flag"]
)

content_df["reason_code"] = "STALE_CONTENT"
content_df["action_label"] = "REFRESH_CONTENT"

ranked_queue = content_df.sort_values(
    "baseline_score",
    ascending=False
)

os.makedirs("work/outputs", exist_ok=True)

ranked_queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print(ranked_queue.head(20))


               client_hash_id           content_hash_id  \
2869  client_05475c07ed21a83a  content_734ce811912af485   
4943  client_06d356715a8ff3b6  content_09063e5eace59644   
4994  client_06d356715a8ff3b6  content_0fe584bb283919e5   
4932  client_06d356715a8ff3b6  content_0792b1fa6ffd28a1   
4958  client_06d356715a8ff3b6  content_0bd6faa72b5c084f   
2151  client_05475c07ed21a83a  content_3f4d2fcadccb8f2f   
3570  client_05475c07ed21a83a  content_a3e9ec895ee10730   
1987  client_05475c07ed21a83a  content_339e5cf69c1278b0   
4635  client_05475c07ed21a83a  content_f05dc68a8522b60c   
4989  client_06d356715a8ff3b6  content_0f76576da13435aa   
2409  client_05475c07ed21a83a  content_51313aa1a62679ea   
2664  client_05475c07ed21a83a  content_647d7186547a8ac4   
2384  client_05475c07ed21a83a  content_4f691ea919daf779   
2668  client_05475c07ed21a83a  content_64bf14213a179731   
1948  client_05475c07ed21a83a  content_30f913a1cd7ee1e4   
2629  client_05475c07ed21a83a  content_6256d596b8647bae 

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

The highest-ranked pages are selected because they are old, have not been updated recently, and may be eligible for optimization.

Possible reasons the recommendation could be wrong:

- The content is evergreen and still performs well.
- The topic is seasonal.
- The content remains accurate despite its age.
- External factors not captured in the warehouse data may influence performance.

In [13]:
top20 = ranked_queue.head(20)

top20[
    [
        "content_hash_id",
        "baseline_score",
        "days_since_update",
        "content_age_days",
        "reason_code",
        "action_label"
    ]
]


,content_hash_id,baseline_score,days_since_update,content_age_days,reason_code,action_label
2869,content_734ce811912af485,0.785542,80,80,STALE_CONTENT,REFRESH_CONTENT
4943,content_09063e5eace59644,0.710542,25,80,STALE_CONTENT,REFRESH_CONTENT
4994,content_0fe584bb283919e5,0.705723,25,79,STALE_CONTENT,REFRESH_CONTENT
4932,content_0792b1fa6ffd28a1,0.695181,20,82,STALE_CONTENT,REFRESH_CONTENT
4958,content_0bd6faa72b5c084f,0.691265,25,76,STALE_CONTENT,REFRESH_CONTENT
2151,content_3f4d2fcadccb8f2f,0.687349,70,70,STALE_CONTENT,REFRESH_CONTENT
3570,content_a3e9ec895ee10730,0.687349,70,70,STALE_CONTENT,REFRESH_CONTENT
1987,content_339e5cf69c1278b0,0.687349,70,70,STALE_CONTENT,REFRESH_CONTENT
4635,content_f05dc68a8522b60c,0.687349,70,70,STALE_CONTENT,REFRESH_CONTENT
4989,content_0f76576da13435aa,0.686446,25,75,STALE_CONTENT,REFRESH_CONTENT


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*



### Weak Picks

Many pages in the Top 20 received very similar scores because they were created and updated during the same time period.

The rule mainly prioritizes pages based on:

- Content age
- Days since last update
- Optimization eligibility

Because these signals are similar across many pages, the ranking may not clearly distinguish the highest-value refresh opportunities.

### Limitation

This baseline rule does not use:

- Traffic data
- Click-through rate (CTR)
- Search ranking position
- User engagement metrics
- Search demand trends

As a result, some pages may be selected for refresh simply because they are old, even if they are still performing well.

### Leakage Check

The baseline score only uses information available at the decision moment:

- content_created_date
- content_updated_date
- optimization_eligible_date

No future performance data was used.

No label-derived columns were used.

No future-window metrics were included in the score.

Therefore, the baseline rule does not contain target leakage.

In [15]:
print("Leakage Check")

features_used = [
    "content_created_date",
    "content_updated_date",
    "optimization_eligible_date"
]

print("Features used:")
for f in features_used:
    print("-", f)

print("\nNo future performance metrics used.")
print("No label-derived columns used.")
print("Leakage check passed.")


Leakage Check
Features used:
- content_created_date
- content_updated_date
- optimization_eligible_date

No future performance metrics used.
No label-derived columns used.
Leakage check passed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.